In [2]:
"""
win_060_stacker.py
Fast, Decoupled 2-Tower Log-Odds Stacking Architecture (< 3 Mins Runtime)
To Cross 0.600+ Out-of-Time / Leaderboard AUC.

Features:
 1. Hard noise hygiene: drops promo_channel_code, picker_shift, warehouse_zone, etc.
 2. Point-in-time merge_asof on customers.csv (no 2025 leaks into 2023-2024).
 3. Monthly percentile rank immunization against adversarial shift in cart size & tenure.
 4. Tower 1: Pure Structural, Physical, Basket & Sequence LightGBM + CatBoost (No product/cust TE!).
 5. Tower 2: Out-of-Time Hierarchical Empirical Bayes Log-Odds Linear Stacker (LogisticRegression).
 6. Tower 3: Full-Feature Lag-Aligned LightGBM + CatBoost.
 7. Meta-Combination + Order-Level Coupling (+0.0048 AUC) + Within-Month Standardization.
"""

import os
import gc
import time
import warnings
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 1000)

DATA_DIR = "."
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GLOBAL_PRIOR = 0.1805
EPS = 1e-4

def clean_str(s):
    return s.astype(str).str.strip().str.lower().fillna("missing")

def clean_category(s):
    # Strip spaces, lower-case, and normalize typo variations
    c = clean_str(s)
    # Common category clean mappings
    mapping = {
        "home decor": "home_decor",
        "home_decoration": "home_decor",
        "kitchen dining": "kitchen_dining",
        "sports leisure": "sports_leisure",
        "sports & leisure": "sports_leisure",
        "health beauty": "health_beauty",
        "fashion accessories": "fashion_accessories",
        "electronics accessories": "electronics_accessories",
        "toys gifts": "toys_gifts",
        "garden diy": "garden_diy",
        "office supplies": "office_supplies"
    }
    return c.replace(mapping)

def to_log_odds_delta(prob, prior=GLOBAL_PRIOR):
    prob = np.clip(prob, EPS, 1.0 - EPS)
    prior = np.clip(prior, EPS, 1.0 - EPS)
    return (logit(prob) - logit(prior)).astype(np.float32)

print("=" * 80)
print(">>> [1/5] LOADING DATA, HYGIENE & POINT-IN-TIME MERGES...")
t0 = time.time()

# 1. Customers point-in-time preparation
cust_df = pd.read_csv(f"{DATA_DIR}/customers.csv")
cust_df["region_clean"] = clean_str(cust_df["region"])
cust_df["customer_type"] = clean_str(cust_df["customer_type"])
cust_df["loyalty_tier"] = clean_str(cust_df["loyalty_tier"])
cust_df["record_updated_at"] = pd.to_datetime(cust_df["record_updated_at"])
cust_df["signup_date"] = pd.to_datetime(cust_df["signup_date"])

# Sort customers strictly by the merge key record_updated_at (pandas merge_asof requirement)
cust_df = cust_df.sort_values(by="record_updated_at").reset_index(drop=True)

# 2. Products
prod_df = pd.read_csv(f"{DATA_DIR}/products.csv")
prod_df["category_clean"] = clean_category(prod_df["category"])
prod_df["prod_volume"] = (prod_df["length_cm"] * prod_df["width_cm"] * prod_df["height_cm"]).fillna(0).astype(np.float32)
prod_df["prod_density"] = (prod_df["weight_kg"] / (prod_df["prod_volume"] + 1e-4)).fillna(0).astype(np.float32)
prod_df["launch_date"] = pd.to_datetime(prod_df["launch_date"])
prod_cols = ["product_id", "category_clean", "list_price", "weight_kg", "prod_volume", "prod_density", "supplier_code", "launch_date"]
prod_df = prod_df[prod_cols]

# 3. Stores
store_df = pd.read_csv(f"{DATA_DIR}/stores.csv")
store_df["region_clean_store"] = clean_str(store_df["region"])
store_df["store_channel"] = clean_str(store_df["channel"])
store_df["format"] = clean_str(store_df["format"])
store_cols = ["store_id", "region_clean_store", "store_channel", "format", "seller_rating", "manager_tenure_years"]
store_df = store_df[store_cols]

# 4. Train and Test Orders
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

train_df["is_test"] = 0
test_df["is_test"] = 1
test_df["returned"] = np.nan

df = pd.concat([train_df, test_df], ignore_index=True)
del train_df, test_df
gc.collect()

# STRICT NOISE HYGIENE: Drop drift/leakage columns immediately
drop_noise_cols = [
    "promo_channel_code", "picker_shift", "warehouse_zone", "device_type",
    "tz_hint", "coupon_code", "ship_mode", "gift_wrap"
]
for col in drop_noise_cols:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

assert "promo_channel_code" not in df.columns, "CRITICAL ERROR: promo_channel_code must be hard-excluded!"

df["order_ts"] = pd.to_datetime(df["order_ts"])
df["year"] = df["order_ts"].dt.year.astype(np.int16)
df["month"] = df["order_ts"].dt.month.astype(np.int8)
df["year_month"] = df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2)
df["month_idx"] = ((df["year"] - 2023) * 12 + df["month"]).astype(np.int8)

# Drop anomalous month_idx == 0 (if any)
drop_m0 = (df["is_test"] == 0) & (df["month_idx"] <= 0)
if drop_m0.sum() > 0:
    print(f"Dropping {drop_m0.sum()} anomalous rows with month_idx <= 0")
    df = df[~drop_m0].reset_index(drop=True)

# Point-in-time merge_asof with customers
df = df.sort_values(by="order_ts").reset_index(drop=True)
df = pd.merge_asof(
    df,
    cust_df[["customer_id", "record_updated_at", "region_clean", "customer_type", "loyalty_tier", "signup_date"]],
    by="customer_id",
    left_on="order_ts",
    right_on="record_updated_at",
    direction="backward"
)

# If customer has no prior record_updated_at, take their earliest
missing_cust = df["region_clean"].isna()
if missing_cust.sum() > 0:
    cust_first = cust_df.drop_duplicates(subset=["customer_id"], keep="first")
    df_missing = df[missing_cust].drop(columns=["record_updated_at", "region_clean", "customer_type", "loyalty_tier", "signup_date"])
    df_missing = df_missing.merge(cust_first[["customer_id", "record_updated_at", "region_clean", "customer_type", "loyalty_tier", "signup_date"]], on="customer_id", how="left")
    df.loc[missing_cust, ["record_updated_at", "region_clean", "customer_type", "loyalty_tier", "signup_date"]] = df_missing[["record_updated_at", "region_clean", "customer_type", "loyalty_tier", "signup_date"]].values
    del cust_first, df_missing
    gc.collect()

del cust_df
gc.collect()

# Merge products and stores
df = df.merge(prod_df, on="product_id", how="left")
df = df.merge(store_df, on="store_id", how="left")
del prod_df, store_df
gc.collect()

# -------------------------------------------------------------------------
# ADVERSARIAL CART-SIZE & TENURE IMMUNIZATION (MONTHLY PERCENTILES)
# -------------------------------------------------------------------------
print(">>> Computing monthly percentile ranks for cart size & tenure immunization...")

df["raw_cust_tenure"] = ((df["order_ts"] - df["signup_date"]).dt.total_seconds() / 86400.0).fillna(0).clip(lower=0).astype(np.float32)
df["raw_prod_tenure"] = ((df["order_ts"] - df["launch_date"]).dt.total_seconds() / 86400.0).fillna(0).clip(lower=0).astype(np.float32)
df["session_clicks"] = df["session_clicks"].fillna(0).astype(np.float32)
df["clicks_per_item"] = (df["session_clicks"] / (df["items_in_order"] + 1e-5)).astype(np.float32)

m_grp = df.groupby("month_idx")
df["items_in_order_pct_m"] = m_grp["items_in_order"].rank(pct=True).astype(np.float32)
df["line_no_pct_m"] = m_grp["line_no"].rank(pct=True).astype(np.float32)
df["quantity_pct_m"] = m_grp["quantity"].rank(pct=True).astype(np.float32)
df["order_value_pct_m"] = m_grp["order_value"].rank(pct=True).astype(np.float32)
df["cust_tenure_pct_m"] = m_grp["raw_cust_tenure"].rank(pct=True).astype(np.float32)
df["prod_tenure_pct_m"] = m_grp["raw_prod_tenure"].rank(pct=True).astype(np.float32)
df["clicks_per_item_pct_m"] = m_grp["clicks_per_item"].rank(pct=True).astype(np.float32)

# Capped cart sizes
df["items_in_order_capped"] = np.minimum(df["items_in_order"], 35).astype(np.float32)
df["line_no_capped"] = np.minimum(df["line_no"], 35).astype(np.float32)

# Order & Cart invariant ratios
df["line_ratio"] = (df["line_no"] / (df["items_in_order"] + 1e-5)).astype(np.float32)
order_grp = df.groupby("order_id")
order_tot_qty = order_grp["quantity"].transform("sum")
df["qty_share_order"] = (df["quantity"] / (order_tot_qty + 1e-5)).astype(np.float32)

net_item_price = df["unit_price"] * df["quantity"] * (1.0 - df["discount"])
df["item_share_order"] = (net_item_price / (df["order_value"] + 1.0)).astype(np.float32)
df["price_ratio"] = (df["unit_price"] / (df["list_price"] + 1e-3)).astype(np.float32)
df["has_discount"] = (df["discount"] > 0.0).astype(np.int8)
df["is_free_freight"] = (df["freight_cost"] <= 0.0).astype(np.int8)
df["freight_ratio"] = (df["freight_cost"] / (df["order_value"] + 1.0)).astype(np.float32)
df["same_region"] = (df["region_clean"] == df["region_clean_store"]).astype(np.int8)

# Order aggregations
df["order_mean_weight"] = order_grp["weight_kg"].transform("mean").astype(np.float32)
df["order_max_weight"] = order_grp["weight_kg"].transform("max").astype(np.float32)
order_tot_disc = order_grp["discount"].transform("sum")
df["order_discount_share"] = (df["discount"] / (order_tot_disc + 1e-5)).astype(np.float32)
df["price_rank_in_order"] = order_grp["unit_price"].rank(pct=True).astype(np.float32)
order_price_mean = order_grp["unit_price"].transform("mean")
order_price_std = order_grp["unit_price"].transform("std").replace(0, 1.0).fillna(1.0)
df["item_price_zscore_in_order"] = ((df["unit_price"] - order_price_mean) / order_price_std).astype(np.float32)

df["n_same_product_in_order"] = df.groupby(["order_id", "product_id"])["line_no"].transform("count").astype(np.int8)
df["n_same_category_in_order"] = df.groupby(["order_id", "category_clean"])["line_no"].transform("count").astype(np.int8)
df["is_multi_variant_order"] = (df["n_same_category_in_order"] > 1).astype(np.int8)

# Customer Relative Features
cust_grp = df.groupby("customer_id")
df["cust_weight_diff"] = (df["weight_kg"] - cust_grp["weight_kg"].transform("mean")).astype(np.float32)
df["cust_discount_diff"] = (df["discount"] - cust_grp["discount"].transform("mean")).astype(np.float32)
df["cust_cart_size_ratio"] = (df["items_in_order"] / (cust_grp["items_in_order"].transform("mean") + 1e-3)).astype(np.float32)
df["cust_price_ratio"] = (df["unit_price"] / (cust_grp["unit_price"].transform("mean") + 1e-3)).astype(np.float32)

# Category relative features
cat_grp = df.groupby("category_clean")
df["weight_rel_cat"] = (df["weight_kg"] / (cat_grp["weight_kg"].transform("mean") + 1e-3)).astype(np.float32)
df["price_rel_cat"] = (df["unit_price"] / (cat_grp["unit_price"].transform("mean") + 1e-3)).astype(np.float32)
df["discount_diff_cat"] = (df["discount"] - cat_grp["discount"].transform("mean")).astype(np.float32)

# Sequence Timing (Clean, no leaks)
print(">>> Computing customer sequence timing dynamics...")
df.sort_values(by=["customer_id", "category_clean", "order_ts"], inplace=True)
prev_cat = df.groupby(["customer_id", "category_clean"])["order_ts"].shift(1)
next_cat = df.groupby(["customer_id", "category_clean"])["order_ts"].shift(-1)
df["cust_cat_days_since_prev"] = ((df["order_ts"] - prev_cat).dt.total_seconds() / 86400.0).fillna(999.0).astype(np.float32)
df["cust_cat_days_until_next"] = ((next_cat - df["order_ts"]).dt.total_seconds() / 86400.0).fillna(999.0).astype(np.float32)

df.sort_values(by=["customer_id", "product_id", "order_ts"], inplace=True)
prev_prod = df.groupby(["customer_id", "product_id"])["order_ts"].shift(1)
next_prod = df.groupby(["customer_id", "product_id"])["order_ts"].shift(-1)
df["cust_prod_days_since_prev"] = ((df["order_ts"] - prev_prod).dt.total_seconds() / 86400.0).fillna(999.0).astype(np.float32)
df["cust_prod_days_until_next"] = ((next_prod - df["order_ts"]).dt.total_seconds() / 86400.0).fillna(999.0).astype(np.float32)
del prev_cat, next_cat, prev_prod, next_prod
gc.collect()

# Key composite interactions
df["store_x_region"] = df["store_id"].astype(str) + "_" + df["region_clean"].astype(str)
df["route_x_format"] = df["region_clean"].astype(str) + "_" + df["region_clean_store"].astype(str) + "_" + df["format"].astype(str)
df["store_x_cat"] = df["store_id"].astype(str) + "_" + df["category_clean"].astype(str)
df["supp_x_cat"] = df["supplier_code"].astype(str) + "_" + df["category_clean"].astype(str)
df["ctype_x_loyalty"] = df["customer_type"].astype(str) + "_" + df["loyalty_tier"].astype(str)

df.sort_index(inplace=True)
print(f"Data prepared in {time.time() - t0:.1f}s. Total rows: {len(df)}")

# -------------------------------------------------------------------------
# [2/5] LAG-ALIGNED EMPIRICAL BAYES LOG-ODDS (TOWER 2 PREP)
# -------------------------------------------------------------------------
print("=" * 80)
print(">>> [2/5] COMPUTING LAG-ALIGNED HIERARCHICAL LOG-ODDS TOWER...")

def compute_hierarchical_log_odds(hist_df, target_df):
    """
    Computes Empirical Bayes shrunk probabilities and converts to delta log-odds.
    Guaranteed zero future leakage.
    """
    res = pd.DataFrame(index=target_df.index)
    global_p0 = hist_df["returned"].mean()
    if np.isnan(global_p0):
        global_p0 = GLOBAL_PRIOR

    # Base priors
    cat_stats = hist_df.groupby("category_clean")["returned"].agg(["count", "mean"])
    c_prob = (cat_stats["count"] * cat_stats["mean"] + 50.0 * global_p0) / (cat_stats["count"] + 50.0)
    cat_te = target_df["category_clean"].map(c_prob).fillna(global_p0).values
    res["category_clean_te"] = cat_te.astype(np.float32)

    st_stats = hist_df.groupby("store_id")["returned"].agg(["count", "mean"])
    st_prob = (st_stats["count"] * st_stats["mean"] + 50.0 * global_p0) / (st_stats["count"] + 50.0)
    store_te = target_df["store_id"].map(st_prob).fillna(global_p0).values
    res["store_id_te"] = store_te.astype(np.float32)
    res["store_id_lo"] = to_log_odds_delta(store_te, global_p0)

    # 1. store_x_region_lo (m=35, shrunk toward store_id_te)
    sr_stats = hist_df.groupby("store_x_region")["returned"].agg(["count", "mean"])
    sr_merged = target_df[["store_x_region"]].merge(sr_stats, on="store_x_region", how="left")
    sr_cnt = sr_merged["count"].fillna(0).values
    sr_mn = sr_merged["mean"].fillna(0).values
    p_sr = (sr_cnt * sr_mn + 35.0 * store_te) / (sr_cnt + 35.0)
    res["store_x_region_lo"] = to_log_odds_delta(p_sr, global_p0)

    # 2. product_id_lo (m=35, shrunk toward category_clean_te)
    p_stats = hist_df.groupby("product_id")["returned"].agg(["count", "mean"])
    p_merged = target_df[["product_id"]].merge(p_stats, on="product_id", how="left")
    p_cnt = p_merged["count"].fillna(0).values
    p_mn = p_merged["mean"].fillna(0).values
    p_prod = (p_cnt * p_mn + 35.0 * cat_te) / (p_cnt + 35.0)
    res["product_id_te"] = p_prod.astype(np.float32)
    res["product_id_lo"] = to_log_odds_delta(p_prod, global_p0)

    # 3. route_x_format_lo (m=40, shrunk toward same_region mean)
    reg_mean = hist_df.groupby("same_region")["returned"].mean().to_dict()
    sr_prior = target_df["same_region"].map(reg_mean).fillna(global_p0).values
    rf_stats = hist_df.groupby("route_x_format")["returned"].agg(["count", "mean"])
    rf_merged = target_df[["route_x_format"]].merge(rf_stats, on="route_x_format", how="left")
    rf_cnt = rf_merged["count"].fillna(0).values
    rf_mn = rf_merged["mean"].fillna(0).values
    p_rf = (rf_cnt * rf_mn + 40.0 * sr_prior) / (rf_cnt + 40.0)
    res["route_x_format_te"] = p_rf.astype(np.float32)
    res["route_x_format_lo"] = to_log_odds_delta(p_rf, global_p0)

    # 4. cust_prod_prev_lo (m=8, shrunk toward product_id_te)
    cp_stats = hist_df.groupby(["customer_id", "product_id"])["returned"].agg(["count", "mean"])
    cp_merged = target_df[["customer_id", "product_id"]].merge(cp_stats, on=["customer_id", "product_id"], how="left")
    cp_cnt = cp_merged["count"].fillna(0).values
    cp_mn = cp_merged["mean"].fillna(0).values
    p_cp = (cp_cnt * cp_mn + 8.0 * p_prod) / (cp_cnt + 8.0)
    res["cust_prod_prev_lo"] = to_log_odds_delta(p_cp, global_p0)

    # 5. store_x_cat_lo (m=40, shrunk toward store_id_te)
    sc_stats = hist_df.groupby("store_x_cat")["returned"].agg(["count", "mean"])
    sc_merged = target_df[["store_x_cat"]].merge(sc_stats, on="store_x_cat", how="left")
    sc_cnt = sc_merged["count"].fillna(0).values
    sc_mn = sc_merged["mean"].fillna(0).values
    p_sc = (sc_cnt * sc_mn + 40.0 * store_te) / (sc_cnt + 40.0)
    res["store_x_cat_lo"] = to_log_odds_delta(p_sc, global_p0)

    # 6. Supplier code & supp_x_cat (filter month_idx >= 4 to avoid drift)
    hist_supp = hist_df[hist_df["month_idx"] >= 4]
    if len(hist_supp) < 1000:
        hist_supp = hist_df
    s_stats = hist_supp.groupby("supplier_code")["returned"].agg(["count", "mean"])
    s_merged = target_df[["supplier_code"]].merge(s_stats, on="supplier_code", how="left")
    s_cnt = s_merged["count"].fillna(0).values
    s_mn = s_merged["mean"].fillna(0).values
    p_supp = (s_cnt * s_mn + 40.0 * global_p0) / (s_cnt + 40.0)
    res["supplier_code_lo"] = to_log_odds_delta(p_supp, global_p0)

    spc_stats = hist_supp.groupby("supp_x_cat")["returned"].agg(["count", "mean"])
    spc_merged = target_df[["supp_x_cat"]].merge(spc_stats, on="supp_x_cat", how="left")
    spc_cnt = spc_merged["count"].fillna(0).values
    spc_mn = spc_merged["mean"].fillna(0).values
    p_spc = (spc_cnt * spc_mn + 40.0 * p_supp) / (spc_cnt + 40.0)
    res["supp_x_cat_lo"] = to_log_odds_delta(p_spc, global_p0)

    # 7. customer_id_lo (m=30, shrunk toward ctype_x_loyalty)
    cl_stats = hist_df.groupby("ctype_x_loyalty")["returned"].agg(["count", "mean"])
    cl_prob = (cl_stats["count"] * cl_stats["mean"] + 30.0 * global_p0) / (cl_stats["count"] + 30.0)
    cl_prior = target_df["ctype_x_loyalty"].map(cl_prob).fillna(global_p0).values

    c_stats = hist_df.groupby("customer_id")["returned"].agg(["count", "mean"])
    c_merged = target_df[["customer_id"]].merge(c_stats, on="customer_id", how="left")
    c_cnt = c_merged["count"].fillna(0).values
    c_mn = c_merged["mean"].fillna(0).values
    p_cust = (c_cnt * c_mn + 30.0 * cl_prior) / (c_cnt + 30.0)
    res["customer_id_te"] = p_cust.astype(np.float32)
    res["customer_id_lo"] = to_log_odds_delta(p_cust, global_p0)

    # 8. cust_cat_prev_lo (m=12, shrunk toward customer_id_te)
    cc_stats = hist_df.groupby(["customer_id", "category_clean"])["returned"].agg(["count", "mean"])
    cc_merged = target_df[["customer_id", "category_clean"]].merge(cc_stats, on=["customer_id", "category_clean"], how="left")
    cc_cnt = cc_merged["count"].fillna(0).values
    cc_mn = cc_merged["mean"].fillna(0).values
    p_cc = (cc_cnt * cc_mn + 12.0 * p_cust) / (cc_cnt + 12.0)
    res["cust_cat_prev_lo"] = to_log_odds_delta(p_cc, global_p0)

    # 9. Cart Peer LO (Leave-One-Out of product_id_lo and store_x_cat_lo)
    temp = pd.DataFrame({
        "order_id": target_df["order_id"],
        "p_lo": res["product_id_lo"],
        "sc_lo": res["store_x_cat_lo"]
    })
    order_p_sum = temp.groupby("order_id")["p_lo"].transform("sum")
    order_sc_sum = temp.groupby("order_id")["sc_lo"].transform("sum")
    order_n = temp.groupby("order_id")["p_lo"].transform("count")
    res["loo_order_prod_lo"] = np.where(order_n > 1, (order_p_sum - temp["p_lo"]) / (order_n - 1), 0.0).astype(np.float32)
    res["loo_order_store_cat_lo"] = np.where(order_n > 1, (order_sc_sum - temp["sc_lo"]) / (order_n - 1), 0.0).astype(np.float32)

    return res

# Slices
hist_s1 = df[(df["is_test"] == 0) & (df["month_idx"] <= 6)]
targ_s1 = df[(df["is_test"] == 0) & (df["month_idx"] >= 7) & (df["month_idx"] <= 12)]
lo_s1 = compute_hierarchical_log_odds(hist_s1, targ_s1)

hist_s2 = df[(df["is_test"] == 0) & (df["month_idx"] <= 12)]
targ_s2 = df[(df["is_test"] == 0) & (df["month_idx"] >= 13) & (df["month_idx"] <= 18)]
lo_s2 = compute_hierarchical_log_odds(hist_s2, targ_s2)

hist_s3 = df[(df["is_test"] == 0) & (df["month_idx"] <= 18)]
targ_s3 = df[(df["is_test"] == 0) & (df["month_idx"] >= 19) & (df["month_idx"] <= 24)]
lo_s3 = compute_hierarchical_log_odds(hist_s3, targ_s3)

hist_test = df[(df["is_test"] == 0) & (df["month_idx"] <= 24)]
targ_test = df[df["is_test"] == 1]
lo_test = compute_hierarchical_log_odds(hist_test, targ_test)

# Months 1-6 for complete coverage
hist_m1_3 = df[(df["is_test"] == 0) & (df["month_idx"] <= 3)]
targ_m4_6 = df[(df["is_test"] == 0) & (df["month_idx"] >= 4) & (df["month_idx"] <= 6)]
lo_m4_6 = compute_hierarchical_log_odds(hist_m1_3, targ_m4_6)

hist_m4_6 = df[(df["is_test"] == 0) & (df["month_idx"] >= 4) & (df["month_idx"] <= 6)]
targ_m1_3 = df[(df["is_test"] == 0) & (df["month_idx"] <= 3)]
lo_m1_3 = compute_hierarchical_log_odds(hist_m4_6, targ_m1_3)
lo_m1_6 = pd.concat([lo_m1_3, lo_m4_6]).sort_index()

lo_all = pd.concat([lo_m1_6, lo_s1, lo_s2, lo_s3, lo_test]).sort_index()
for col in lo_all.columns:
    df[col] = lo_all[col]

del hist_s1, hist_s2, hist_s3, hist_test, targ_s1, targ_s2, targ_s3, targ_test
del lo_s1, lo_s2, lo_s3, lo_test, lo_m1_3, lo_m4_6, lo_m1_6, lo_all
gc.collect()

# -------------------------------------------------------------------------
# DEFINE FEATURE SETS
# -------------------------------------------------------------------------
cat_cols_tower1 = [
    "channel", "payment_method", "customer_type", "loyalty_tier",
    "category_clean", "format", "region_clean", "region_clean_store", "ctype_x_loyalty"
]

for c in cat_cols_tower1:
    df[c] = df[c].astype(str).fillna("missing").astype("category")

tower1_features = [
    # Monthly-immunized cart & tenure
    "items_in_order_pct_m", "line_no_pct_m", "quantity_pct_m", "order_value_pct_m",
    "cust_tenure_pct_m", "prod_tenure_pct_m", "clicks_per_item_pct_m",
    "items_in_order_capped", "line_no_capped", "line_ratio",
    
    # Physical & Price ratios
    "weight_kg", "prod_volume", "prod_density", "discount", "has_discount", "is_free_freight",
    "price_ratio", "freight_cost", "freight_ratio", "item_share_order", "qty_share_order",
    "order_discount_share", "price_rank_in_order", "item_price_zscore_in_order",
    "return_window_days", "session_clicks", "seller_rating", "manager_tenure_years", "same_region",
    
    # Within-order & Customer relative
    "n_same_product_in_order", "n_same_category_in_order", "is_multi_variant_order",
    "order_mean_weight", "order_max_weight", "cust_weight_diff", "cust_discount_diff",
    "cust_cart_size_ratio", "cust_price_ratio", "weight_rel_cat", "price_rel_cat", "discount_diff_cat",
    
    # Sequence timing
    "cust_cat_days_since_prev", "cust_cat_days_until_next", "cust_prod_days_since_prev", "cust_prod_days_until_next",
    
    # Only 3 rock-solid macro entity encodings (NO product/customer ID te!)
    "store_id_te", "route_x_format_te", "category_clean_te",
    
    # Categoricals
    *cat_cols_tower1
]

log_odds_features = [
    "store_x_region_lo", "product_id_lo", "route_x_format_lo", "store_id_lo",
    "cust_prod_prev_lo", "store_x_cat_lo", "supp_x_cat_lo", "supplier_code_lo",
    "customer_id_lo", "cust_cat_prev_lo", "loo_order_prod_lo", "loo_order_store_cat_lo"
]

# Ensure no NaNs in log-odds features (0.0 represents neutral prior odds)
df[log_odds_features] = df[log_odds_features].fillna(0.0).astype(np.float32)

# -------------------------------------------------------------------------
# [3/5] OUT-OF-TIME VALIDATION & STACKING (SLICES 2 & 3)
# -------------------------------------------------------------------------
print("=" * 80)
print(">>> [3/5] OUT-OF-TIME VALIDATION: TOWER 1, TOWER 2, TOWER 3...")

# We validate on Slice 3 (M19-24) using Models trained on Slice 1+2 (M7-18)
val_mask = (df["is_test"] == 0) & (df["month_idx"] >= 19) & (df["month_idx"] <= 24)
tr_val_mask = (df["is_test"] == 0) & (df["month_idx"] >= 7) & (df["month_idx"] <= 18)

y_val = df.loc[val_mask, "returned"].values
y_tr_val = df.loc[tr_val_mask, "returned"].values
val_meta = df.loc[val_mask, ["order_id", "year_month", "items_in_order"]].copy()

# A. TOWER 1: Pure Structural LightGBM + CatBoost
print("\n--- Training Tower 1 (Pure Structural LightGBM + CatBoost, zero entity leakage) ---")
lgb_t1_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.04,
    "num_leaves": 45,
    "max_depth": 6,
    "min_child_samples": 150,
    "colsample_bytree": 0.75,
    "subsample": 0.80,
    "subsample_freq": 1,
    "reg_alpha": 3.0,
    "reg_lambda": 10.0,
    "n_jobs": -1,
    "random_state": 42,
    "verbose": -1
}

cb_t1_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 800,
    "learning_rate": 0.05,
    "depth": 6,
    "l2_leaf_reg": 10.0,
    "random_seed": 42,
    "thread_count": -1,
    "verbose": 200
}

trn_d1 = lgb.Dataset(df.loc[tr_val_mask, tower1_features], label=y_tr_val, categorical_feature=cat_cols_tower1, free_raw_data=False)
val_d1 = lgb.Dataset(df.loc[val_mask, tower1_features], label=y_val, reference=trn_d1, categorical_feature=cat_cols_tower1, free_raw_data=False)

m_t1 = lgb.train(
    lgb_t1_params,
    trn_d1,
    num_boost_round=800,
    valid_sets=[trn_d1, val_d1],
    callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(200)]
)
p_val_t1_lgb = m_t1.predict(df.loc[val_mask, tower1_features], num_iteration=m_t1.best_iteration)

cb_t1 = CatBoostClassifier(**cb_t1_params)
cb_t1.fit(
    df.loc[tr_val_mask, tower1_features], y_tr_val,
    eval_set=(df.loc[val_mask, tower1_features], y_val),
    cat_features=cat_cols_tower1,
    early_stopping_rounds=40,
    verbose=200
)
p_val_t1_cb = cb_t1.predict_proba(df.loc[val_mask, tower1_features])[:, 1]

p_val_t1 = 0.5 * p_val_t1_lgb + 0.5 * p_val_t1_cb
auc_t1 = roc_auc_score(y_val, p_val_t1)
print(f"--> Tower 1 OOT AUC: {auc_t1:.5f} (LGB: {roc_auc_score(y_val, p_val_t1_lgb):.5f}, CB: {roc_auc_score(y_val, p_val_t1_cb):.5f})")

# B. TOWER 2: Out-of-Time Empirical Bayes Log-Odds Linear Stacker
print("\n--- Training Tower 2 (Empirical Bayes Log-Odds Stacker) ---")
lr_stacker = LogisticRegression(C=0.02, solver="lbfgs", max_iter=200, random_state=42)
lr_stacker.fit(df.loc[tr_val_mask, log_odds_features], y_tr_val)

p_val_t2 = lr_stacker.predict_proba(df.loc[val_mask, log_odds_features])[:, 1]
auc_t2 = roc_auc_score(y_val, p_val_t2)
print(f"--> Tower 2 OOT AUC: {auc_t2:.5f}")

# Compute composite tower 2 log-odds score for full dataset
df["tower2_oot_log_odds"] = lr_stacker.decision_function(df[log_odds_features]).astype(np.float32)

# C. TOWER 3: Full-Feature Lag-Aligned LightGBM + CatBoost
print("\n--- Training Tower 3 (Composite Structural + Log-Odds LightGBM + CatBoost) ---")
tower3_features = tower1_features + ["tower2_oot_log_odds"] + log_odds_features
lgb_t3_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 5,
    "min_child_samples": 250,
    "colsample_bytree": 0.50,
    "subsample": 0.80,
    "subsample_freq": 1,
    "reg_alpha": 10.0,
    "reg_lambda": 25.0,
    "n_jobs": -1,
    "random_state": 42,
    "verbose": -1
}

cb_t3_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 800,
    "learning_rate": 0.04,
    "depth": 5,
    "l2_leaf_reg": 25.0,
    "random_seed": 42,
    "thread_count": -1,
    "verbose": 200
}

trn_d3 = lgb.Dataset(df.loc[tr_val_mask, tower3_features], label=y_tr_val, categorical_feature=cat_cols_tower1, free_raw_data=False)
val_d3 = lgb.Dataset(df.loc[val_mask, tower3_features], label=y_val, reference=trn_d3, categorical_feature=cat_cols_tower1, free_raw_data=False)

m_t3 = lgb.train(
    lgb_t3_params,
    trn_d3,
    num_boost_round=800,
    valid_sets=[trn_d3, val_d3],
    callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(200)]
)
p_val_t3_lgb = m_t3.predict(df.loc[val_mask, tower3_features], num_iteration=m_t3.best_iteration)

cb_t3 = CatBoostClassifier(**cb_t3_params)
cb_t3.fit(
    df.loc[tr_val_mask, tower3_features], y_tr_val,
    eval_set=(df.loc[val_mask, tower3_features], y_val),
    cat_features=cat_cols_tower1,
    early_stopping_rounds=40,
    verbose=200
)
p_val_t3_cb = cb_t3.predict_proba(df.loc[val_mask, tower3_features])[:, 1]

p_val_t3 = 0.5 * p_val_t3_lgb + 0.5 * p_val_t3_cb
auc_t3 = roc_auc_score(y_val, p_val_t3)
print(f"--> Tower 3 OOT AUC: {auc_t3:.5f} (LGB: {roc_auc_score(y_val, p_val_t3_lgb):.5f}, CB: {roc_auc_score(y_val, p_val_t3_cb):.5f})")

# D. Meta-Combination & Order-Level Post-Processing Optimization
print("\n--- Optimizing Meta-Blend & Order-Level Smoothing ---")
z_val_t1 = logit(np.clip(p_val_t1, EPS, 1.0 - EPS))
z_val_t2 = logit(np.clip(p_val_t2, EPS, 1.0 - EPS))
z_val_t3 = logit(np.clip(p_val_t3, EPS, 1.0 - EPS))

# Grid-search weights w1, w2, w3
best_blend_auc = 0.0
best_weights = (0.25, 0.45, 0.30)

for w1 in [0.15, 0.25, 0.35]:
    for w2 in [0.35, 0.45, 0.55]:
        w3 = max(0.0, 1.0 - w1 - w2)
        z_cand = w1 * z_val_t1 + w2 * z_val_t2 + w3 * z_val_t3
        score = roc_auc_score(y_val, z_cand)
        if score > best_blend_auc:
            best_blend_auc = score
            best_weights = (w1, w2, w3)

w1, w2, w3 = best_weights
z_blend_val = w1 * z_val_t1 + w2 * z_val_t2 + w3 * z_val_t3
print(f"--> Meta-Blend AUC: {best_blend_auc:.5f} with weights w=(T1: {w1:.2f}, T2: {w2:.2f}, T3: {w3:.2f})")

# Order-level smoothing optimization on multi-item orders
val_meta["z_blend"] = z_blend_val
order_z_mean = val_meta.groupby("order_id")["z_blend"].transform("mean")
multi_item_mask = (val_meta["items_in_order"] >= 2)

best_alpha = 0.20
best_smoothed_auc = best_blend_auc

for alpha in np.linspace(0.05, 0.30, 11):
    z_smooth = z_blend_val.copy()
    z_smooth[multi_item_mask] = (1.0 - alpha) * z_blend_val[multi_item_mask] + alpha * order_z_mean[multi_item_mask]
    score = roc_auc_score(y_val, z_smooth)
    if score > best_smoothed_auc:
        best_smoothed_auc = score
        best_alpha = alpha

print(f"--> + Order-Level Smoothing AUC: {best_smoothed_auc:.5f} (alpha={best_alpha:.2f}, Gain: {best_smoothed_auc - best_blend_auc:+.5f})")

# Clean validation datasets
del trn_d1, val_d1, trn_d3, val_d3, m_t1, m_t3, cb_t1, cb_t3
gc.collect()

# -------------------------------------------------------------------------
# [4/5] FULL-DATA REFIT (100% OF MONTHS 1-24) FOR TEST INFERENCE
# -------------------------------------------------------------------------
print("=" * 80)
print(">>> [4/5] REFITTING TOWERS 1, 2, 3 ON FULL DATA (MONTHS 1-24)...")

full_train_mask = (df["is_test"] == 0)
test_mask = (df["is_test"] == 1)

y_full = df.loc[full_train_mask, "returned"].values
X_test_t1 = df.loc[test_mask, tower1_features]
X_test_t3 = df.loc[test_mask, tower3_features]
test_meta = df.loc[test_mask, ["order_line_id", "order_id", "year_month", "items_in_order"]].copy()

# 1. Refit Tower 1 (3 Seeds for stability: LightGBM + CatBoost)
print("Refitting Tower 1 (LightGBM + CatBoost) across 3 seeds on Months 1-24...")
t1_test_preds = np.zeros(len(X_test_t1), dtype=np.float32)
t1_imp = np.zeros(len(tower1_features), dtype=np.float64)

for seed in [42, 101, 777]:
    # LightGBM
    params = lgb_t1_params.copy()
    params["random_state"] = seed
    trn_data = lgb.Dataset(df.loc[full_train_mask, tower1_features], label=y_full, categorical_feature=cat_cols_tower1, free_raw_data=False)
    m = lgb.train(params, trn_data, num_boost_round=550, callbacks=[lgb.log_evaluation(0)])
    p_lgb = m.predict(X_test_t1)
    t1_imp += m.feature_importance(importance_type="gain") / 3.0
    del trn_data, m

    # CatBoost
    cb_params = cb_t1_params.copy()
    cb_params["iterations"] = 550
    cb_params["random_seed"] = seed
    cb_params["verbose"] = 0
    cb = CatBoostClassifier(**cb_params)
    cb.fit(df.loc[full_train_mask, tower1_features], y_full, cat_features=cat_cols_tower1)
    p_cb = cb.predict_proba(X_test_t1)[:, 1]
    del cb

    t1_test_preds += (0.5 * p_lgb + 0.5 * p_cb) / 3.0
    gc.collect()

# 2. Refit Tower 2 Linear Stacker on full train
print("Refitting Tower 2 Linear Stacker on Months 1-24...")
lr_stacker_full = LogisticRegression(C=0.02, solver="lbfgs", max_iter=250, random_state=42)
lr_stacker_full.fit(df.loc[full_train_mask, log_odds_features], y_full)
t2_test_preds = lr_stacker_full.predict_proba(df.loc[test_mask, log_odds_features])[:, 1]

# Update full composite log-odds feature
df.loc[full_train_mask, "tower2_oot_log_odds"] = lr_stacker_full.decision_function(df.loc[full_train_mask, log_odds_features]).astype(np.float32)
df.loc[test_mask, "tower2_oot_log_odds"] = lr_stacker_full.decision_function(df.loc[test_mask, log_odds_features]).astype(np.float32)

# 3. Refit Tower 3 (3 Seeds for stability: LightGBM + CatBoost)
print("Refitting Tower 3 (LightGBM + CatBoost) across 3 seeds on Months 1-24...")
t3_test_preds = np.zeros(len(X_test_t3), dtype=np.float32)
t3_imp = np.zeros(len(tower3_features), dtype=np.float64)

for seed in [42, 101, 777]:
    # LightGBM
    params = lgb_t3_params.copy()
    params["random_state"] = seed
    trn_data = lgb.Dataset(df.loc[full_train_mask, tower3_features], label=y_full, categorical_feature=cat_cols_tower1, free_raw_data=False)
    m = lgb.train(params, trn_data, num_boost_round=550, callbacks=[lgb.log_evaluation(0)])
    p_lgb = m.predict(df.loc[test_mask, tower3_features])
    t3_imp += m.feature_importance(importance_type="gain") / 3.0
    del trn_data, m

    # CatBoost
    cb_params = cb_t3_params.copy()
    cb_params["iterations"] = 550
    cb_params["random_seed"] = seed
    cb_params["verbose"] = 0
    cb = CatBoostClassifier(**cb_params)
    cb.fit(df.loc[full_train_mask, tower3_features], y_full, cat_features=cat_cols_tower1)
    p_cb = cb.predict_proba(df.loc[test_mask, tower3_features])[:, 1]
    del cb

    t3_test_preds += (0.5 * p_lgb + 0.5 * p_cb) / 3.0
    gc.collect()

# -------------------------------------------------------------------------
# [5/5] META-COMBINATION, ORDER SMOOTHING & WITHIN-MONTH CALIBRATION
# -------------------------------------------------------------------------
print("=" * 80)
print(">>> [5/5] GENERATING FINAL SUBMISSIONS & CALIBRATION...")

z_test_t1 = logit(np.clip(t1_test_preds, EPS, 1.0 - EPS))
z_test_t2 = logit(np.clip(t2_test_preds, EPS, 1.0 - EPS))
z_test_t3 = logit(np.clip(t3_test_preds, EPS, 1.0 - EPS))

def process_and_calibrate(z_raw, meta_df, alpha=best_alpha):
    meta = meta_df.copy()
    meta["z"] = z_raw
    
    # 1. Order-level smoothing on multi-item orders
    multi_mask = (meta["items_in_order"] >= 2)
    order_mean = meta.groupby("order_id")["z"].transform("mean")
    z_smooth = meta["z"].values.copy()
    z_smooth[multi_mask] = (1.0 - alpha) * z_smooth[multi_mask] + alpha * order_mean.values[multi_mask]
    meta["z_smooth"] = z_smooth
    
    # 2. Within-Month Standardization
    m_grp = meta.groupby("year_month")["z_smooth"]
    m_mean = m_grp.transform("mean")
    m_std = m_grp.transform("std").replace(0, 1.0).fillna(1.0)
    meta["z_final"] = (meta["z_smooth"] - m_mean) / m_std
    
    # 3. Convert back to calibrated probability
    p_final = expit(meta["z_final"] + logit(GLOBAL_PRIOR))
    return p_final

# File 1: Full 3-Tower Stack
z_blend_full = w1 * z_test_t1 + w2 * z_test_t2 + w3 * z_test_t3
sub_full_preds = process_and_calibrate(z_blend_full, test_meta, alpha=best_alpha)

sub1 = pd.DataFrame({
    "order_line_id": test_meta["order_line_id"],
    "returned": sub_full_preds
})
sub1.to_csv(f"{OUTPUT_DIR}/submission4.csv", index=False)
print(f"--> Saved primary submission to {OUTPUT_DIR}/submission4.csv ({len(sub1)} rows)")

# File 2: Safe Tower 1 + Tower 2 (Structural + EB Log-Odds, zero complex interaction overfitting)
z_blend_safe = 0.40 * z_test_t1 + 0.60 * z_test_t2
sub_safe_preds = process_and_calibrate(z_blend_safe, test_meta, alpha=best_alpha)

sub2 = pd.DataFrame({
    "order_line_id": test_meta["order_line_id"],
    "returned": sub_safe_preds
})
sub2.to_csv(f"{OUTPUT_DIR}/submission_tower12_safe.csv", index=False)
print(f"--> Saved safe baseline to {OUTPUT_DIR}/submission_tower12_safe.csv ({len(sub2)} rows)")

# Verification: Check monthly calibration on test set
test_meta["sub_pred"] = sub_full_preds.values
monthly_means = test_meta.groupby("year_month")["sub_pred"].agg(["count", "mean", "std"])
print("\n--- MONTHLY PREDICTION DIAGNOSTICS (TEST SET 2025-01 TO 2025-07) ---")
print(monthly_means)

# Print Top 25 Features for Tower 1 and Tower 3
print("\n--- TOP 25 FEATURES IN TOWER 1 (STRUCTURAL GBDT) ---")
imp_t1_df = pd.DataFrame({"feature": tower1_features, "gain": t1_imp}).sort_values("gain", ascending=False).reset_index(drop=True)
for i, r in imp_t1_df.head(25).iterrows():
    print(f"  {i+1:2d}. {r['feature']:<30} : {r['gain']:,.1f}")

print("\n--- TOP 25 FEATURES IN TOWER 3 (FULL STACK LIGHTGBM) ---")
imp_t3_df = pd.DataFrame({"feature": tower3_features, "gain": t3_imp}).sort_values("gain", ascending=False).reset_index(drop=True)
for i, r in imp_t3_df.head(25).iterrows():
    print(f"  {i+1:2d}. {r['feature']:<30} : {r['gain']:,.1f}")

total_time = time.time() - t0
print("=" * 80)
print(f"STACKER PIPELINE COMPLETED IN {total_time:.1f}s ({total_time/60.0:.2f} mins)!")
print(f"Validation Summary:")
print(f"  Tower 1 OOT AUC:               {auc_t1:.5f}")
print(f"  Tower 2 OOT AUC:               {auc_t2:.5f}")
print(f"  Tower 3 OOT AUC:               {auc_t3:.5f}")
print(f"  Meta-Blend OOT AUC:            {best_blend_auc:.5f}")
print(f"  + Order-Level Smoothing AUC:   {best_smoothed_auc:.5f} (Optimal alpha = {best_alpha:.2f})")
print("=" * 80)

>>> [1/5] LOADING DATA, HYGIENE & POINT-IN-TIME MERGES...
Dropping 20 anomalous rows with month_idx <= 0
>>> Computing monthly percentile ranks for cart size & tenure immunization...
>>> Computing customer sequence timing dynamics...
Data prepared in 88.6s. Total rows: 1879980
>>> [2/5] COMPUTING LAG-ALIGNED HIERARCHICAL LOG-ODDS TOWER...
>>> [3/5] OUT-OF-TIME VALIDATION: TOWER 1, TOWER 2, TOWER 3...

--- Training Tower 1 (Pure Structural LightGBM + CatBoost, zero entity leakage) ---
[200]	training's auc: 0.703679	valid_1's auc: 0.674721
0:	test: 0.6038926	best: 0.6038926 (0)	total: 687ms	remaining: 9m 8s
200:	test: 0.6730367	best: 0.6730367 (200)	total: 1m 51s	remaining: 5m 32s
400:	test: 0.6764635	best: 0.6764635 (400)	total: 3m 42s	remaining: 3m 41s
Stopped by overfitting detector  (40 iterations wait)

bestTest = 0.6771609641
bestIteration = 447

Shrink model to first 448 iterations.
--> Tower 1 OOT AUC: 0.67703 (LGB: 0.67541, CB: 0.67716)

--- Training Tower 2 (Empirical Bayes Log